In [0]:
%sql
USE CATALOG databricks_proyecto_jhon;
-- Creamos el schema
CREATE SCHEMA IF NOT EXISTS lakehouse;
-- Creamos el volumen
CREATE VOLUME IF NOT EXISTS lakehouse.datos_crudos;

In [0]:
from pyspark.sql import functions as F

# Guardamos en una variable la ruta exacta de tu archivo crudo. El protocolo abfss:// es el estándar de seguridad nativo de Azure para leer datos directamente desde tu Azure Data Lake Storage Gen2.
RUTA_LANDING_BCRP = "abfss://lakehouse@datalakejhon2026.dfs.core.windows.net/datos_crudos/landing/tipo_cambio_bcrp.json"

# Lee el archivo desde el Data Lake y lo convierte en un DataFrame (una tabla en memoria). La instrucción option("multiline", "true") es crítica aquí: por defecto, Spark espera que los JSON tengan un registro por línea. Como las APIs suelen entregar JSONs formateados con saltos de línea y anidaciones, esta opción le permite a Spark leer la estructura completa sin romperse.

df_crudo_bcrp = spark.read.option("multiline", "true").json(RUTA_LANDING_BCRP)

# Toma los datos crudos y les agrega dos nuevas columnas de metadatos usando el comando .withColumn(): 
# _fecha_carga: Utiliza F.current_timestamp() para estampar la fecha y hora exactas en las que el servidor procesó el archivo.
# _origen: Utiliza F.lit() para inyectar un valor "literal" o fijo. En este caso, escribe "API_BCRP" en todas las filas para que, si en el futuro mezclas múltiples fuentes, sepas exactamente de dónde vino esta fila.

df_bronce_bcrp = (df_crudo_bcrp
    .withColumn("_fecha_carga", F.current_timestamp())
    .withColumn("_origen", F.lit("API_BCRP"))
)

# Guardado en formato Delta (Capa Bronce)
#.format("delta"): Guarda los datos usando la tecnología Delta Lake, garantizando transacciones seguras (ACID) y compresión avanzada.
#.mode("overwrite"): Le indica que si la tabla ya existe de una ejecución anterior, la reemplace por completo con estos nuevos datos.
#.option("overwriteSchema", "true"): Activa la evolución de esquema. Si mañana el BCRP agrega una nueva columna al JSON, Spark no fallará; simplemente actualizará la estructura de tu tabla automáticamente.
#.saveAsTable(...): No solo guarda los archivos físicos, sino que registra oficialmente la tabla en tu Unity Catalog (catálogo.esquema.tabla).

(df_bronce_bcrp.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true") 
    .saveAsTable("databricks_proyecto_jhon.lakehouse.bronce_tipo_cambio_bcrp")
)

print(f"✅ BCRP guardado en Bronce: {df_bronce_bcrp.count()} registros.")

display(df_bronce_bcrp)

In [0]:
from pyspark.sql import functions as F

# 1. Datos simulados ampliados (Casos exitosos + Errores intencionales)
datos_simulados = [
    # --- Datos correctos (Para graficar en Power BI) ---
    (1, 101, 501, 150.50, "2024-03-01", "COMPLETADA"),
    (2, 101, 502, 200.00, "2024-03-01", "COMPLETADA"),
    (6, 102, 503, 450.00, "2024-03-02", "COMPLETADA"),
    (7, 103, 504, 120.00, "2024-03-03", "COMPLETADA"),
    (8, 101, 501, 340.50, "2024-03-04", "COMPLETADA"),
    (9, 102, 505, 990.00, "2024-03-04", "COMPLETADA"),
    (10, 103, 502, 15.00, "2024-03-05", "COMPLETADA"),
    (11, 101, 501, 210.00, "2024-03-06", "COMPLETADA"),
    (12, 102, 503, 530.00, "2024-03-07", "COMPLETADA"),
    (13, 103, 506, 100.00, "2024-03-08", "COMPLETADA"),
    (14, 101, 504, 85.00, "2024-03-08", "COMPLETADA"),
    (15, 102, 501, 400.00, "2024-03-09", "COMPLETADA"),
    
    # --- Datos con errores (Para que la Capa Plata los limpie) ---
    (3, 102, 501, -50.00, "2024-03-02", "COMPLETADA"),  # ERROR: Monto negativo
    (4, 103, 503, 300.00, None, "COMPLETADA"),          # ERROR: Sin fecha
    (5, 101, 504, 120.00, "2024-03-03", "CANCELADA"),   # ERROR: Venta cancelada
    (16, 101, None, 500.00, "2024-03-10", "COMPLETADA"),# ERROR: Sin cliente
    (17, 102, 505, 0.00, "2024-03-10", "COMPLETADA")    # ERROR: Monto cero
]
columnas = ["id", "id_empleado", "id_cliente", "monto", "fecha", "estado"]
df_crudo_ventas = spark.createDataFrame(datos_simulados, columnas)

# 2. Enriquecimiento de Auditoría (El "Sello de agua")
df_bronce_ventas = (df_crudo_ventas
    .withColumn("_fecha_carga", F.current_timestamp())
    .withColumn("_origen", F.lit("SISTEMA_VENTAS_MOCK"))
)

# 3. Carga a Delta Lake (Registro en Unity Catalog)
(df_bronce_ventas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("databricks_proyecto_jhon.lakehouse.bronce_ventas")
)

print(f"✅ ¡Ventas guardadas en Bronce! Total: {df_bronce_ventas.count()} registros.")
display(df_bronce_ventas)